In [1]:
import os
import cv2
import time
import pytz
import numpy as np
import pandas as pd
from datetime import datetime
import face_recognition

In [2]:
def read_image(path):
    img=cv2.imread(path)#reading the image
    (h,w)=img.shape[:2]
    width=500
    ratio=width/float(w)
    height=int(h*ratio)
    return cv2.resize(img,(width,height))#resizing the image

In [3]:
known_encodings=[]
known_names=[]
known_dir='known_images'#path of directory which contain the images 
for i in os.listdir(known_dir):
    if i!='.ipynb_checkpoints':
        img=read_image(known_dir+'/'+i)
        img_enc=face_recognition.face_encodings(img)[0]
        known_encodings.append(img_enc)
        known_names.append(i.split('.')[0])

In [5]:
import sqlite3
#creating or connecting to previous database
conn=sqlite3.connect('Attendance2.db')
cursor=conn.cursor()
#creating table
cursor.execute("""create table Present_list1(
USN text primary key,
Name text,
Date text,
Time text
)""")
USNS=[7,8,18,48,10]#these are unique values for each image
face_classifier=cv2.CascadeClassifier('C:\\Users\\Lenovo\\OneDrive\\Documents\\haarcascade_frontalface_default.xml')
#path of haarcascade_frontalface file to find face in the image
video_capture=cv2.VideoCapture(0)
n=1
present=[]
attend_list=[]
import time
while True:
    xx=0
    _,frame=video_capture.read()
    face=face_classifier.detectMultiScale(frame,1.3,5)
    if len(face)==0:
        cv2.putText(frame,'No Face Found',(10,30),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,255),2)
    else:
        for (x,y,w,h)in face:
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
            if n%15==0:
                xx=1
                try:
                    img_enc=face_recognition.face_encodings(frame)[0]
                    results=face_recognition.compare_faces(known_encodings,img_enc)
                    distance=face_recognition.face_distance(known_encodings,img_enc)
                    mi=min(distance)
                    ind=list(distance).index(mi)
                    usn=USNS[ind]
                    if mi>0.50:
                        cv2.putText(frame,'Face Not Matched',(10,30),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,255),2)
                    else:
                        if usn in present:
                            cv2.putText(frame,known_names[ind]+' Marked',(10,30),cv2.FONT_HERSHEY_COMPLEX,1,(0,127,255),2)
                        else:
                            present.append(usn)
                            tim=datetime.now(pytz.timezone('Asia/kolkata'))
                            l=(tim.strftime('%Y-%m-%d %H:%M:%S'))
                            date,time=l.split(' ')
                            ind_time=time.split(':')
                            if int(ind_time[0])>12:
                                pm_time=str(int(ind_time[0])-12)+':'+ind_time[1]+':'+ind_time[2]
                                attend_list.append((USNS[ind],known_names[ind],date,pm_time))
                                cursor.execute("insert into Present_list1 values(:USN,:Name,:Date,:Time)",
                                        {
                                            'USN':USNS[ind],
                                            'Name':known_names[ind],
                                            'Date':date,
                                            'Time':pm_time
                                                    })
                            else:
                                attend_list.append((USNS[ind],known_names[ind],date,time))
                                cursor.execute("insert into Present_list1 values(:USN,:Name,:Date,:Time)",
                                        {
                                            'USN':USNS[ind],
                                            'Name':known_names[ind],
                                            'Date':date,
                                            'Time':time
                                                    })
                            cv2.putText(frame,known_names[ind],(10,30),cv2.FONT_HERSHEY_COMPLEX,1,(0,255,0),2)
                except Exception as e:
                    pass
        n+=1
    cv2.putText(frame,'q->Exit',(450,30),cv2.FONT_HERSHEY_COMPLEX,1,(0,0,255),2)        
    cv2.imshow('Attendance',frame)
    if xx==1:
        cv2.waitKey(1000)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
video_capture.release()
cv2.destroyAllWindows()
df=pd.DataFrame(attend_list,columns=['USN','NAME','DATE','TIME'])
ndf=df.sort_values(by=['USN'],ascending=True)
ndf.index=[i for i in range(1,len(ndf)+1)]
print(ndf)
conn.commit()

conn.close()

   USN   NAME        DATE      TIME
1    7  Dhoni  2024-05-01  12:55:18
2   18  Kohli  2024-05-01  12:53:56


In [6]:
conn=sqlite3.connect('Attendance2.db')
cursor=conn.cursor()
cursor.execute('select *,oid from present_list1')
records=cursor.fetchall()#fetchone,fetchmany(10)
print_record=''
for record in records:
    print_record+=str(record)+"\n"
    print(record)
conn.commit()

conn.close()

('18', 'Kohli', '2024-05-01', '12:53:56', 1)
('7', 'Dhoni', '2024-05-01', '12:55:18', 2)
